In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()

In [3]:
df_data = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet("../data/aggregation/1979-2025")

In [4]:
df_data.show()

+---------------+--------------+---------------+---------------+-----------------+-------------+------------------+----------+-----------+----+---------+-----------+----------------+------------------+-----------------------+-------------------+
|collision_index|casualty_class|sex_of_casualty|age_of_casualty|casualty_severity|casualty_type|collision_severity|      date|day_of_week|time|road_type|speed_limit|light_conditions|weather_conditions|road_surface_conditions|urban_or_rural_area|
+---------------+--------------+---------------+---------------+-----------------+-------------+------------------+----------+-----------+----+---------+-----------+----------------+------------------+-----------------------+-------------------+
|  197901A1DAK71|             1|              1|             24|                3|          109|                 3|1979-01-01|          2|   3|        9|         30|               4|                 8|                      3|                 -1|
|  197901A1DAK71

In [5]:
# Handle missing value

In [6]:
# Settings null value

In [7]:
null_col_set = [
    "road_type",
    "casualty_class",
    "sex_of_casualty",
    "age_of_casualty",
    "speed_limit",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "urban_or_rural_area",
]

In [8]:
for col in null_col_set:
    df_data = df_data.withColumn(col, F.when((F.col(col) == -1), None).otherwise(F.col(col)))

df_data = df_data \
    .withColumn("sex_of_casualty", F.when((F.col("sex_of_casualty") == -1) | (F.col("sex_of_casualty") == 9), None).otherwise(F.col("sex_of_casualty")))

In [9]:
# Check missing value
def check_missing_val(df):
    check_list = []
    
    for col_name, dtype in df.dtypes:
        condition = F.col(col_name).isNull()
        
        if dtype in ['double', 'float']:
            condition = condition | F.isnan(F.col(col_name))
    
        check_list.append(F.count(F.when(condition, col_name)).alias(col_name))
    df.select(*check_list).show(vertical=True)

In [10]:
check_missing_val(df_data)

-RECORD 0--------------------------
 collision_index         | 0       
 casualty_class          | 1       
 sex_of_casualty         | 11940   
 age_of_casualty         | 218553  
 casualty_severity       | 0       
 casualty_type           | 0       
 collision_severity      | 0       
 date                    | 10      
 day_of_week             | 0       
 time                    | 10      
 road_type               | 124     
 speed_limit             | 358     
 light_conditions        | 2621    
 weather_conditions      | 3350    
 road_surface_conditions | 17010   
 urban_or_rural_area     | 4837106 



In [11]:
# Handle missing value drop null value
df_clean = df_data.dropna(subset=['casualty_class', 'date', 'time', 'road_type'])

In [12]:
# Handle duplicate value
df_clean = df_clean.dropDuplicates(["collision_index","casualty_class","age_of_casualty", "sex_of_casualty"])

In [13]:
check_missing_val(df_clean)

-RECORD 0--------------------------
 collision_index         | 0       
 casualty_class          | 0       
 sex_of_casualty         | 10843   
 age_of_casualty         | 207894  
 casualty_severity       | 0       
 casualty_type           | 0       
 collision_severity      | 0       
 date                    | 0       
 day_of_week             | 0       
 time                    | 0       
 road_type               | 0       
 speed_limit             | 356     
 light_conditions        | 2497    
 weather_conditions      | 3257    
 road_surface_conditions | 16754   
 urban_or_rural_area     | 4779321 



In [14]:
# Handle missing value with median for speed and age
median_speed = df_clean.approxQuantile("speed_limit", [0.5], 0.01)[0]
median_age = df_clean.approxQuantile("age_of_casualty", [0.5], 0.01)[0]

In [15]:
df_clean = df_clean.fillna({
    "speed_limit": median_speed,
    "age_of_casualty": median_age
})

In [16]:
# Handle missing value with mode for sex, light, weather and road surface
mode_cols = [
    'sex_of_casualty',
    'light_conditions',
    'weather_conditions',
    'road_surface_conditions'
]

In [17]:
mode_values = df_clean.agg(*[F.mode(c).alias(c) for c in mode_cols]).collect()[0].asDict()

In [18]:
df_clean = df_clean.fillna(mode_values)

In [19]:
check_missing_val(df_clean)

-RECORD 0--------------------------
 collision_index         | 0       
 casualty_class          | 0       
 sex_of_casualty         | 0       
 age_of_casualty         | 0       
 casualty_severity       | 0       
 casualty_type           | 0       
 collision_severity      | 0       
 date                    | 0       
 day_of_week             | 0       
 time                    | 0       
 road_type               | 0       
 speed_limit             | 0       
 light_conditions        | 0       
 weather_conditions      | 0       
 road_surface_conditions | 0       
 urban_or_rural_area     | 4779321 



In [20]:
df_clean = df_clean.fillna({"urban_or_rural_area": 3})

In [21]:
check_missing_val(df_clean)

-RECORD 0----------------------
 collision_index         | 0   
 casualty_class          | 0   
 sex_of_casualty         | 0   
 age_of_casualty         | 0   
 casualty_severity       | 0   
 casualty_type           | 0   
 collision_severity      | 0   
 date                    | 0   
 day_of_week             | 0   
 time                    | 0   
 road_type               | 0   
 speed_limit             | 0   
 light_conditions        | 0   
 weather_conditions      | 0   
 road_surface_conditions | 0   
 urban_or_rural_area     | 0   



In [22]:
df_clean.select("time").show()

+----+
|time|
+----+
|   4|
|  20|
|   0|
|  17|
|   9|
|   8|
|  19|
|  22|
|   0|
|  16|
|  19|
|  13|
|  11|
|  23|
|  13|
|  15|
|  19|
|   7|
|  15|
|   6|
+----+
only showing top 20 rows


In [23]:
# Binning col time into 4 bins
# 1: morning, 2: noon, 3: afternoon, 4: night

df_clean = df_clean.withColumn("time_bin",
        F.when((F.col("time") >=5 ) & (F.col("time") < 11), 1) # 1: Morning
        .when((F.col("time") >=11 ) & (F.col("time") < 14), 2) # 2: Noon
        .when((F.col("time") >=14 ) & (F.col("time") < 18), 3) # 3: Afternoon
        .otherwise(4)
)

In [24]:
df_clean.select("time_bin").show()

+--------+
|time_bin|
+--------+
|       4|
|       4|
|       4|
|       3|
|       1|
|       1|
|       4|
|       4|
|       4|
|       3|
|       4|
|       2|
|       2|
|       4|
|       2|
|       3|
|       4|
|       1|
|       3|
|       1|
+--------+
only showing top 20 rows


In [25]:
# Encoded data using StringIndexer

In [26]:
categorical_cols = [
    "road_type",
    "casualty_class",
    "sex_of_casualty",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "casualty_type",
    "urban_or_rural_area",
    "time_bin"
]

In [27]:
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
    for col in categorical_cols
]

In [28]:
pipeline = Pipeline(stages=indexers)
df_clean_encoded = pipeline.fit(df_clean).transform(df_clean)

In [29]:
df_clean_encoded.select(
    "time_bin", "time_bin_idx", 
    "sex_of_casualty", "sex_of_casualty_idx"
).show(5)

+--------+------------+---------------+-------------------+
|time_bin|time_bin_idx|sex_of_casualty|sex_of_casualty_idx|
+--------+------------+---------------+-------------------+
|       4|         0.0|              2|                1.0|
|       4|         0.0|              2|                1.0|
|       4|         0.0|              1|                0.0|
|       3|         1.0|              1|                0.0|
|       1|         2.0|              1|                0.0|
+--------+------------+---------------+-------------------+
only showing top 5 rows


In [30]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Cột nhãn
label_col = "casualty_severity"

# Các cột categorical đã được mã hóa ở bước 3
encoded_feature_cols = [
    "road_type_idx",
    "casualty_class_idx",
    "sex_of_casualty_idx",
    "light_conditions_idx",
    "weather_conditions_idx",
    "road_surface_conditions_idx",
    "urban_or_rural_area_idx",
    "time_bin_idx",
    "casualty_type_idx",
    "day_of_week"
]

# Các cột số
numeric_feature_cols = [
    "age_of_casualty",
    "speed_limit"
]

# Gộp tất cả cột feature
feature_cols = numeric_feature_cols + encoded_feature_cols

# Tránh trường hợp label bị nằm trong feature
feature_cols = [c for c in feature_cols if c != label_col]

print("Danh sách feature dùng cho mô hình:")
for c in feature_cols:
    print("-", c)

Danh sách feature dùng cho mô hình:
- age_of_casualty
- speed_limit
- road_type_idx
- casualty_class_idx
- sex_of_casualty_idx
- light_conditions_idx
- weather_conditions_idx
- road_surface_conditions_idx
- urban_or_rural_area_idx
- time_bin_idx
- casualty_type_idx
- day_of_week


In [31]:
df_outlier = df_clean_encoded

# Các cột số cần loại ngoại lai
outlier_cols = [
    "age_of_casualty",
    "speed_limit"
]

# Lưu ngưỡng IQR của từng cột
outlier_bounds = {}

for col_name in outlier_cols:
    q1, q3 = df_outlier.approxQuantile(col_name, [0.25, 0.75], 0.01)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_bounds[col_name] = {
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound
    }

    print(f"\nCột: {col_name}")
    print(f"Q1 = {q1}")
    print(f"Q3 = {q3}")
    print(f"IQR = {iqr}")
    print(f"Lower bound = {lower_bound}")
    print(f"Upper bound = {upper_bound}")

# Đếm số dòng trước khi lọc
before_outlier_count = df_outlier.count()
print("\nSố dòng trước khi loại ngoại lai:", before_outlier_count)

# Lọc ngoại lai lần lượt theo từng cột
df_no_outlier = df_outlier

for col_name in outlier_cols:
    lower_bound = outlier_bounds[col_name]["lower_bound"]
    upper_bound = outlier_bounds[col_name]["upper_bound"]

    df_no_outlier = df_no_outlier.filter(
        (F.col(col_name) >= lower_bound) & (F.col(col_name) <= upper_bound)
    )

# Đếm số dòng sau khi lọc
after_outlier_count = df_no_outlier.count()
print("Số dòng sau khi loại ngoại lai:", after_outlier_count)
print("Số dòng bị loại:", before_outlier_count - after_outlier_count)


Cột: age_of_casualty
Q1 = 20.0
Q3 = 44.0
IQR = 24.0
Lower bound = -16.0
Upper bound = 80.0

Cột: speed_limit
Q1 = 30.0
Q3 = 60.0
IQR = 30.0
Lower bound = -15.0
Upper bound = 105.0

Số dòng trước khi loại ngoại lai: 11900959
Số dòng sau khi loại ngoại lai: 11721072
Số dòng bị loại: 179887


In [32]:
# Kiểm tra lại min/max sau khi loại ngoại lai
for col_name in outlier_cols:
    print(f"\nKiểm tra lại cột: {col_name}")
    df_no_outlier.select(
        F.min(col_name).alias("min_value"),
        F.max(col_name).alias("max_value")
    ).show()


Kiểm tra lại cột: age_of_casualty
+---------+---------+
|min_value|max_value|
+---------+---------+
|        0|       80|
+---------+---------+


Kiểm tra lại cột: speed_limit
+---------+---------+
|min_value|max_value|
+---------+---------+
|        0|       99|
+---------+---------+



In [33]:
df_balance_input = df_no_outlier

# Kiểm tra phân bố lớp trước khi cân bằng
print("Phân bố lớp trước khi cân bằng:")
df_balance_input.groupBy(label_col).count().orderBy(label_col).show()

# Tính số lượng từng lớp
class_count_df = df_balance_input.groupBy(label_col).count()

# Lấy số lượng lớn nhất
max_class_count = class_count_df.agg(F.max("count")).collect()[0][0]
print("Số lượng lớp lớn nhất:", max_class_count)

# Tạo trọng số cho từng lớp
class_weight_df = class_count_df.withColumn(
    "class_weight",
    F.lit(max_class_count) / F.col("count")
)

print("Trọng số của từng lớp:")
class_weight_df.orderBy(label_col).show()

# Join trọng số vào dataframe chính
df_weighted = df_balance_input.join(
    class_weight_df.select(label_col, "class_weight"),
    on=label_col,
    how="left"
)

# Kiểm tra dữ liệu sau khi thêm class_weight
df_weighted.select(label_col, "class_weight").show(10)

Phân bố lớp trước khi cân bằng:
+-----------------+-------+
|casualty_severity|  count|
+-----------------+-------+
|                1| 145389|
|                2|1874759|
|                3|9700924|
+-----------------+-------+

Số lượng lớp lớn nhất: 9700924
Trọng số của từng lớp:
+-----------------+-------+-----------------+
|casualty_severity|  count|     class_weight|
+-----------------+-------+-----------------+
|                1| 145389|66.72391996643488|
|                2|1874759|5.174491227939164|
|                3|9700924|              1.0|
+-----------------+-------+-----------------+

+-----------------+-----------------+
|casualty_severity|     class_weight|
+-----------------+-----------------+
|                1|66.72391996643488|
|                3|              1.0|
|                3|              1.0|
|                3|              1.0|
|                3|              1.0|
|                3|              1.0|
|                3|              1.0|
|             

In [34]:
df_weighted.select("age_of_casualty").show()

+---------------+
|age_of_casualty|
+---------------+
|             26|
|             40|
|             25|
|             43|
|             46|
|             13|
|             20|
|             63|
|             17|
|             19|
|              9|
|             19|
|             33|
|             43|
|             57|
|             19|
|             46|
|             21|
|             30|
|             56|
+---------------+
only showing top 20 rows


In [35]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

df_scale_input = df_weighted

numeric_assembler = VectorAssembler(
    inputCols=numeric_feature_cols,
    outputCol="numeric_features_before_scaling",
    handleInvalid="keep"
)

df_numeric_vector = numeric_assembler.transform(df_scale_input)

numeric_scaler = StandardScaler(
    inputCol="numeric_features_before_scaling",
    outputCol="numeric_features_scaled",
    withMean=True,
    withStd=True
)

numeric_scaler_model = numeric_scaler.fit(df_numeric_vector)
df_numeric_scaled = numeric_scaler_model.transform(df_numeric_vector)

final_assembler = VectorAssembler(
    inputCols=["numeric_features_scaled"] + encoded_feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

df_final = final_assembler.transform(df_numeric_scaled)

df_final.select(
    label_col,
    "class_weight",
    "numeric_features_before_scaling",
    "numeric_features_scaled",
    "features"
).show(5, truncate=False)

+-----------------+-----------------+-------------------------------+-----------------------------------------+---------------------------------------------------------------------------------+
|casualty_severity|class_weight     |numeric_features_before_scaling|numeric_features_scaled                  |features                                                                         |
+-----------------+-----------------+-------------------------------+-----------------------------------------+---------------------------------------------------------------------------------+
|3                |1.0              |[26.0,30.0]                    |[-0.3620901251604634,-0.6844095677216478]|[-0.3620901251604634,-0.6844095677216478,4.0,2.0,1.0,1.0,2.0,1.0,0.0,0.0,2.0,1.0]|
|3                |1.0              |[40.0,30.0]                    |[0.4383922831406438,-0.6844095677216478] |[0.4383922831406438,-0.6844095677216478,4.0,1.0,1.0,1.0,2.0,0.0,0.0,0.0,0.0,7.0] |
|3                |1.0        

In [44]:
import pyspark.sql.functions as F

ratio = {1.0: 0.8, 2.0: 0.8, 3.0: 0.8}

df_train = df_clean_encoded.sampleBy(label_col, fractions=ratio, seed=42)
df_test = df_clean_encoded.subtract(df_train)

df_train.groupBy(label_col).count().orderBy(label_col).show()

+-----------------+-------+
|casualty_severity|  count|
+-----------------+-------+
|                1| 124741|
|                2|1535575|
|                3|7861403|
+-----------------+-------+



In [51]:
df_train.count()

9521719

In [50]:
from pyspark.ml.stat import ChiSquareTest
import pandas as pd

chi_assembler = VectorAssembler(
    inputCols=encoded_feature_cols, 
    outputCol="cat_features_vector", 
    handleInvalid="keep"
)

df_for_chi = chi_assembler.transform(df_train).select("cat_features_vector", label_col)
chi_result = ChiSquareTest.test(df_for_chi, "cat_features_vector", label_col).head()

p_values = chi_result.pValues.toArray()
statistics = chi_result.statistics.toArray()

chi_df = pd.DataFrame({
    "Đặc trưng (Feature)": encoded_feature_cols,
    "Điểm Chi-Square (Statistic)": statistics,
    "P-Value": p_values
})

chi_df = chi_df.sort_values(by="Điểm Chi-Square (Statistic)", ascending=False)
print(chi_df.to_string(index=False))

        Đặc trưng (Feature)  Điểm Chi-Square (Statistic)  P-Value
          casualty_type_idx                375319.512134      0.0
    urban_or_rural_area_idx                132751.980204      0.0
         casualty_class_idx                126312.083856      0.0
       light_conditions_idx                 66535.206065      0.0
        sex_of_casualty_idx                 60265.601488      0.0
              road_type_idx                 35250.322576      0.0
               time_bin_idx                 35143.512183      0.0
     weather_conditions_idx                  8736.860050      0.0
                day_of_week                  6954.815127      0.0
road_surface_conditions_idx                  1294.159972      0.0
